In [21]:
import json
import pandas as pd
import re
from pathlib import Path


In [22]:
# Path to the trajectory JSON (change as needed)
JSON_PATH = "react-Qwen3-4B-Instruct-2507-0.0_range_0--1_user-hosted_vllm_Qwen_Qwen3-4B-Instruct-2507-llm_0209172831.json"
JSON_PATH = "react-Qwen3-4B-Instruct-2507-0.0_range_0--1_user-hosted_vllm_Qwen_Qwen3-4B-Instruct-2507-llm_0209191055.json"

with open(JSON_PATH) as f:
    data = json.load(f)

print(f"Loaded {len(data)} task trajectories")
print(f"Keys per entry: {list(data[0].keys())}")

Loaded 25 task trajectories
Keys per entry: ['task_id', 'reward', 'info', 'traj', 'trial']


In [23]:
# High-level summary DataFrame
def extract_actions_from_traj(traj):
    """Extract action names from 'Action:\n{"name": "..."}' in assistant content."""
    names = []
    for msg in traj:
        if msg.get("role") != "assistant":
            continue
        content = msg.get("content") or ""
        # Match Action:\n{"name": "tool_name", ...}
        for m in re.finditer(r'"name"\s*:\s*"([^"]+)"', content):
            names.append(m.group(1))
    return names

rows = []
for item in data:
    info = item.get("info") or {}
    task = info.get("task") or {}
    reward_info = info.get("reward_info") or {}
    actions = reward_info.get("actions") or task.get("actions") or []
    reward_inner = reward_info.get("info") or {}
    task_actions = task.get("actions") or []
    task_action_names = [a.get("name") for a in task_actions if isinstance(a, dict) and "name" in a]
    traj = item.get("traj") or []
    traj_action_names = extract_actions_from_traj(traj)
    rows.append({
        "task_id": item.get("task_id"),
        "reward": item.get("reward"),
        "task_actions": task_action_names,
        "traj_actions": traj_action_names,
        "num_gt_actions": len(actions),
        "r_actions": reward_inner.get("r_actions"),
        "traj_len": len(traj),
        "trial": item.get("trial", 0),
    })

df = pd.DataFrame(rows)
df

,task_id,reward,task_actions,traj_actions,num_gt_actions,r_actions,traj_len,trial
0,0,0.0,"[send_certificate, cancel_reservation, search_...","[get_user_details, get_reservation_details, th...",3,0.0,44,0
1,1,0.0,"[get_onestop_flights, get_onestop_flights]","[think, think, think, think, think, think, thi...",2,NaN,62,0
2,2,0.0,"[send_certificate, cancel_reservation]","[get_user_details, think, think, think, think,...",2,NaN,62,0
3,3,0.0,"[get_direct_flights, get_onestop_flights, get_...","[search_onestop_flight, book_reservation, thin...",5,NaN,62,0
4,4,1.0,"[get_onestop_flights, get_reservation_details]","[search_onestop_flight, respond, think, respon...",2,1.0,24,0
5,5,0.0,"[get_reservation_details, get_reservation_deta...","[get_reservation_details, think, think, think,...",2,NaN,62,0
6,6,0.0,"[send_certificate, book_reservation]","[get_user_details, send_certificate, respond, ...",2,0.0,20,0
7,7,0.0,"[send_certificate, book_reservation]","[send_certificate, respond, think, search_dire...",2,0.0,16,0
8,8,1.0,[get_direct_flights],"[search_direct_flight, respond, think, respond...",1,1.0,16,0
9,9,0.0,"[get_direct_flights, get_user_details]","[get_user_details, search_direct_flight, book_...",2,0.0,10,0


In [ ]:
# Reward and success stats
print("Reward distribution:")
print(df["reward"].value_counts().sort_index())
print(f"\nSuccess rate (reward == 1.0): {df['reward'].eq(1.0).mean():.2%}")
print(f"Mean reward: {df['reward'].mean():.4f}")

Reward distribution:
reward
0.0    19
1.0     6
Name: count, dtype: int64

Success rate (reward == 1.0): 24.00%
Mean reward: 0.2400


In [ ]:
# Trajectory length and number of ground-truth actions
print("Trajectory length (messages per task):")
print(df["traj_len"].describe())
print("\nNumber of ground-truth actions per task:")
print(df["num_gt_actions"].describe())

Trajectory length (messages per task):
count    25.000000
mean     33.360000
std      23.120842
min       4.000000
25%      16.000000
50%      22.000000
75%      62.000000
max      62.000000
Name: traj_len, dtype: float64

Number of ground-truth actions per task:
count    25.000000
mean      2.720000
std       1.594783
min       0.000000
25%       2.000000
50%       2.000000
75%       3.000000
max       8.000000
Name: num_gt_actions, dtype: float64


In [26]:
# Parse tool/action names from assistant messages in each traj (optional)
def extract_actions_from_traj(traj):
    """Extract action names from 'Action:\n{"name": "..."}' in assistant content."""
    names = []
    for msg in traj:
        if msg.get("role") != "assistant":
            continue
        content = msg.get("content") or ""
        # Match Action:\n{"name": "tool_name", ...}
        for m in re.finditer(r'"name"\s*:\s*"([^"]+)"', content):
            names.append(m.group(1))
    return names

# Add agent action counts for first few tasks as example
df["agent_actions"] = [extract_actions_from_traj(item.get("traj", [])) for item in data]
df["num_agent_actions"] = df["agent_actions"].apply(len)
print("Agent action count (parsed from traj) vs ground-truth:")
print(df[["task_id", "num_gt_actions", "num_agent_actions"]].head(10))

Agent action count (parsed from traj) vs ground-truth:
   task_id  num_gt_actions  num_agent_actions
0        0               3                 21
1        1               2                 30
2        2               2                 30
3        3               5                 30
4        4               2                 11
5        5               2                 30
6        6               2                  9
7        7               2                  7
8        8               1                  7
9        9               2                  4


In [27]:
# Compare ground truth actions vs agent actions
# If traj_actions column doesn't exist, extract it on the fly
if 'traj_actions' not in df.columns:
    print("Warning: 'traj_actions' column not found. Extracting from trajectories...")
    df['traj_actions'] = df.apply(
        lambda row: extract_actions_from_traj(data[row.name].get('traj', [])), 
        axis=1
    )

for idx, row in df.iterrows():
    print(f"\n{'='*60}")
    print(f"Task ID: {row['task_id']} | Reward: {row['reward']}")
    print(f"{'='*60}")
    print(f"Ground Truth Actions: {row['task_actions']}")
    print(f"Agent Actions:        {row['traj_actions']}")
    print(f"Match: {row['task_actions'] == row['traj_actions']}")
    if row['task_actions'] != row['traj_actions']:
        print(f"  GT count: {len(row['task_actions'])}, Agent count: {len(row['traj_actions'])}")


Task ID: 0 | Reward: 0.0
Ground Truth Actions: ['send_certificate', 'cancel_reservation', 'search_onestop_flight']
Agent Actions:        ['get_user_details', 'get_reservation_details', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'transfer_to_human_agents']
Match: False
  GT count: 3, Agent count: 21

Task ID: 1 | Reward: 0.0
Ground Truth Actions: ['get_onestop_flights', 'get_onestop_flights']
Agent Actions:        ['think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think']
Match: False
  GT count: 2, Agent count: 30

Task ID: 2 | Reward: 0.0
Ground Truth Actions: ['send_certificate', 'cancel_reservation']
Agent Actions:        ['get_user_details', 'think', 'thin

In [ ]:
# Tool usage distribution (which tools the agent called)
from collections import Counter
all_agent_tools = []
for names in df["agent_actions"]:
    all_agent_tools.extend(names)
print("Tool call counts (agent):")
for name, count in Counter(all_agent_tools).most_common():
    print(f"  {name}: {count}")

Tool call counts (agent):
  think: 238
  respond: 86
  book_reservation: 28
  get_reservation_details: 18
  search_onestop_flight: 15
  get_user_details: 14
  search_direct_flight: 10
  transfer_to_human_agents: 7
  cancel_reservation: 4
  send_certificate: 3
  update_reservation_baggages: 1
  list_all_airports: 1
  update_reservation_flights: 1
